# Vachan V2 spike — a Hinglish *tone dial* with control vectors

**What this proves:** you can shift a model's tone (formal English ↔ casual Hinglish) by *adding a direction to its hidden states at generation time* — **no retraining, no fine-tuning**. This is the V2 path beyond prompt-steering.

**How it works (plain English):** a control vector is a single direction in the model's "thought space". We find it by showing the model the *same* situation described two ways — once as a casual Hinglish friend, once as a formal corporate email — and subtracting the two internal states. That difference *is* the tone axis. At generation we add `coeff × vector`: `+` pushes Hinglish, `-` pushes formal English, `0` is the untouched model.

**n8n analogy:** the model is a pipeline; the control vector is a slider node we splice into the middle layers that nudges every token toward one tone.

> Runtime: ~5–10 min on a Kaggle **T4** (set Accelerator → GPU T4 first). Llama-3.1-8B in 4-bit ≈ 6 GB, fits one T4.

## 0. Setup (read me)

1. **Kaggle**: top-right **⋮ → Accelerator → GPU T4 x2** (or just T4). Also **Internet: On** (Settings).
2. **Model**: we use a **non-gated** Llama-3.1-8B mirror, so you need **no HuggingFace token**. (If you'd rather use the official `meta-llama/Llama-3.1-8B-Instruct`, accept its license on HF, make a read token, add it in Kaggle **Add-ons → Secrets** as `HF_TOKEN`, and flip `MODEL` below.)
3. Run cells top to bottom (**Run All**).
4. **Two gotchas, both normal:** the install cell prints red "dependency conflict" warnings — expected (it repairs numpy; see the note in that cell). And don't re-run the model-load cell by itself — it loads a *second* copy of the model and runs the T4 out of memory. To try new dial settings, re-run only the **Turn the dial** cell.

In [ ]:
# repeng = the control-vector library (extract + inject). The rest are standard.
# Installing repeng pins numpy<2, but Kaggle's prebuilt PyTorch is compiled against
# numpy 2.x — the mismatch throws "numpy.dtype size changed" on import. The second
# line repairs numpy back to 2.x. The red "dependency conflict" warnings both lines
# print are EXPECTED and harmless (we override repeng's old pin via a patch below).
!pip install -q repeng transformers accelerate bitsandbytes
!pip install -q --force-reinstall "numpy>=2.0,<2.3"

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # faster shard download (preinstalled on Kaggle)

import torch
import numpy as np
if not hasattr(np, "float_"):
    np.float_ = np.float64  # repeng still uses this pre-numpy-2.0 alias; re-add it so import works
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from repeng import ControlVector, ControlModel, DatasetEntry

# Non-gated mirror — no HF login needed. Same weights as Meta's official release.
MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token_id = tokenizer.eos_token_id

# 4-bit so 8B fits a single 16 GB T4 (≈6 GB of weights).
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")

# Wrap a band of middle/late layers — these are where 'style' lives. ControlModel
# lets us inject a vector into exactly these layers at generation time. This band
# (-5 to -18) is tuned for 8B's 32 layers; wider tends to garble the formal side.
model = ControlModel(model, list(range(-5, -18, -1)))
print("loaded:", MODEL)

## 1. Define the tone axis with contrastive pairs

We hand `repeng` many `(positive, negative)` pairs that are identical *except* for the tone instruction. It reads the model's hidden state for each and learns the single direction that separates them. The short generic suffixes just give it many token positions to read from — more positions → a cleaner vector.

In [ ]:
# The two ends of the dial:
POS = "You are texting a close Indian friend on WhatsApp: warm, casual, code-mix Hindi and English (Hinglish), short and natural."
NEG = "You are writing a formal corporate email: polished, professional, pure English, complete sentences."

# Generic continuations — the vector is read at each of these positions.
SUFFIXES = [
    "", "I", "I think", "I think we", "Let me", "Sure", "Honestly", "Okay so",
    "The plan", "We should", "It is", "Yeah", "Right now", "Tomorrow", "That", "So",
]

def framed(persona: str, suffix: str) -> str:
    msgs = [
        {"role": "system", "content": persona},
        {"role": "user", "content": "Give me an update on the project."},
    ]
    s = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return s + suffix

dataset = [DatasetEntry(positive=framed(POS, s), negative=framed(NEG, s)) for s in SUFFIXES]
print(len(dataset), "contrastive pairs")
print("--- one positive example (tail) ---")
print(dataset[2].positive[-160:])

## 2. Extract the control vector

One pass over the pairs — no gradients, no training loop. This is why it's cheap.

In [ ]:
model.reset()  # make sure no previous control is active
hinglish_vector = ControlVector.train(model, tokenizer, dataset)

_layers = list(hinglish_vector.directions.keys())
print("control vector trained over", len(_layers), "layers")
print("per-layer direction shape:", hinglish_vector.directions[_layers[0]].shape)

## 3. Turn the dial — same prompt, several tones

`coeff > 0` → push toward Hinglish; `coeff < 0` → push toward formal English; `0` → the untouched model. On this 8B the usable range is roughly **−4 (clean formal English)** through **+4.5 (Hinglish flavour)**; past ±5 the text starts to break down. If the tone shifts with the number, the thesis holds.

In [ ]:
def generate(prompt: str, coeff: float) -> str:
    model.reset()
    if coeff != 0:
        model.set_control(hinglish_vector, coeff)
    # tokenize=False -> get plain text, then tokenize separately. Newer transformers
    # returns a BatchEncoding from apply_chat_template(return_tensors=...), which
    # model.generate won't accept directly — so pass input_ids/attention_mask explicitly.
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=80,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    model.reset()
    return tokenizer.decode(out[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

# Range tuned for THIS 8B: formal English stays clean down to about -5; the Hinglish
# side shows up around +4.5 and garbles past +5 (base Llama-3.1-8B isn't Hinglish-
# native — see the notes below). 0 is the untouched model.
PROMPT = "Can you give me an update on the deployment?"
for c in [-4.0, 0.0, 4.0, 4.5]:
    print(f"\n=========== coeff {c:+} ===========", flush=True)
    print(generate(PROMPT, c), flush=True)

## 4. Read the result + what's next

**What actually happens on this 8B (measured):**
- `-4` → clean, structured **formal English** ("The most recent updates are as follows: …").
- `0` → the untouched model, neutral.
- `+4.5` → tone bends toward **Hinglish** — Hindi words start leaking in and grammar loosens ("I apology, but I neither *kee* track…"). Push to `+5` and it tips into full (garbled) Hinglish; past that, token soup.

So the dial demonstrably moves tone with the number — **the thesis holds.** The asymmetry (formal side clean, Hinglish side only appearing right at the garble threshold) is a **model limit, not a mechanism one**: base Llama-3.1-8B is English-native with no stable, clean Hinglish mode to land in. It can be *pushed* toward Hinglish but not *rested* there.

**Tuning knobs:**
- Text degenerates at high `|coeff|` → back off toward ±4.
- Cleaner Hinglish → either (a) train the vector from **real parallel Hinglish↔English sentences** instead of an English persona label, or (b) swap to a Hinglish-native base (e.g. `sarvamai/OpenHathi-7B-Hi-v0.1-Base`). Deferred — the English dial is enough for now.
- Effect too weak → widen the layer band (`range(-3, -22, -1)`), but on 8B that tends to garble the formal side before it helps.

**Path into Vachan (later, not this notebook):**
1. Per-persona vectors: build the contrast from the persona's *own* anchors (Hinglish vs their English-translated anchors — we already store both) → a personalized tone dial.
2. Serve it: this needs *our* forward pass (Groq can't inject vectors). Run this 8B (or Sarvam-30B on 2×T4) behind vLLM/transformers on a serverless GPU, only for high-value personas that the PFS gate keeps failing — exactly the documented Path-B trigger.
3. The Fidelity Ring's neural cosine becomes the objective we tune the coeff against.